In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import os
import joblib  # Used to save the encoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

# 1. Set the experiment name
mlflow.set_experiment("AgriPrice_Prediction")

def train_agropulse():
    with mlflow.start_run():
        # --- Updated Data Path ---
        # Points to the 'Dataset' folder
        data_path = os.path.join('Dataset', 'merged_commodities.csv')
        df = pd.read_csv(data_path)
        
        # Simple cleaning
        df = df.dropna(subset=['RetailUnitPrice', 'Commodity', 'Market'])
        
        # Features & Target
        categorical_cols = ['Commodity', 'Classification', 'Market', 'County']
        
        # --- Encode & Save Encoder (For MLOps Production) ---
        encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
            
        # Save the encoders as an 'artifact' so we can use them later
        joblib.dump(encoders, "label_encoders.pkl")
        mlflow.log_artifact("label_encoders.pkl")
            
        X = df[categorical_cols]
        y = df['RetailUnitPrice']
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # --- Model Params ---
        n_estimators = 100
        max_depth = 10
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # --- Training ---
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # --- Evaluation ---
        predictions = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2_score", r2)

        # --- Save the Model ---
        mlflow.sklearn.log_model(model, "price_prediction_model")
        
        print(f"Success! Model trained using data from {data_path}")
        print(f"Metrics - RMSE: {rmse:.2f}, R2: {r2:.2f}")

if __name__ == "__main__":
    train_agropulse()

2026/03/03 09:31:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/03 09:31:16 INFO mlflow.store.db.utils: Updating database tables
2026/03/03 09:31:18 INFO mlflow.tracking.fluent: Experiment with name 'AgriPrice_Prediction' does not exist. Creating a new experiment.
2026/03/03 09:31:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 09:31:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Success! Model trained using data from Dataset\merged_commodities.csv
Metrics - RMSE: 9271.17, R2: 0.77
